# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [2]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "art": "../datasets/art.csv",
    "art_2": "../datasets/art_2.csv",
    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 art 테이블

행 수: 11
컬럼: ['id', '분야', '단과대학', '학과(전공)']

첫 5개 행:
   id   분야   단과대학       학과(전공)
0   1  디자인  디자인대학  커뮤니케이션디자인전공
1   2  디자인  디자인대학      패션디자인전공
2   3  디자인  디자인대학    텍스타일디자인전공
3   4  디자인  디자인대학    스페이스디자인전공
4   5  디자인  디자인대학  인더스트리얼디자인전공

데이터 타입:
id        int64
분야          str
단과대학        str
학과(전공)      str
dtype: object


📋 art_2 테이블

행 수: 20
컬럼: ['id', '분야', '신청 교과목명', '학점', '개설학년', '인정시기', '비고']

첫 5개 행:
   id  분야           신청 교과목명  학점  개설학년       인정시기  비고
0   1  공통          문화예술교육개론   2     2  2018년 1학기 NaN
1   2  공통  문화예술교육현장의 이해와 실습   2     3  2017년 2학기 NaN
2   3  연극         연극영화교과교육론   3     3  2018년 1학기 NaN
3   4  연극      연극영화교과교재및연구법   3     3  2017년 1학기 NaN
4   5  연극        연극교육프로그램개발   2     4  2018년 1학기 NaN

데이터 타입:
id           int64
분야             str
신청 교과목명        str
학점           int64
개설학년         int64
인정시기           str
비고         float64
dtype: object



✓ 총 2개의 테이블 로드 완료


## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [6]:
# TODO: 각 테이블의 주요 통계를 확인하세요
# 예시:
# - 특정 컬럼의 고유값 개수
# - 카테고리별 데이터 분포
# - 결측치 확인

for art, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {art} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    print(df.info())

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # TODO: 팀 데이터에 맞는 추가 탐색 코드를 작성하세요
    # 예시:
    # print("\n[카테고리 분포]")
    # print(df['YOUR_COLUMN'].value_counts())


📊 art 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      11 non-null     int64
 1   분야      11 non-null     str  
 2   단과대학    11 non-null     str  
 3   학과(전공)  11 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1011.0 bytes
None

[결측치]
결측치 없음

📊 art_2 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       20 non-null     int64  
 1   분야       20 non-null     str    
 2   신청 교과목명  20 non-null     str    
 3   학점       20 non-null     int64  
 4   개설학년     20 non-null     int64  
 5   인정시기     20 non-null     str    
 6   비고       0 non-null      float64
dtypes: float64(1), int64(3), str(3)
memory usage: 2.4 KB
None

[결측치]
비고    20
dtype: int64


## 3. Supabase PostgreSQL 연결

In [7]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['art', 'art_2', 'departments', 'office_floors', 'organizations']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [8]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE art (
	id BIGINT, 
	"분야" TEXT, 
	"단과대학" TEXT, 
	"학과(전공)" TEXT
)

/*
3 rows from art table:
id	분야	단과대학	학과(전공)
1	디자인	디자인대학	커뮤니케이션디자인전공
2	디자인	디자인대학	패션디자인전공
3	디자인	디자인대학	텍스타일디자인전공
*/


CREATE TABLE art_2 (
	id BIGINT, 
	"분야" TEXT, 
	"신청 교과목명" TEXT, 
	"학점" BIGINT, 
	"개설학년" BIGINT, 
	"인정시기" TEXT, 
	"비고" TEXT
)

/*
3 rows from art_2 table:
id	분야	신청 교과목명	학점	개설학년	인정시기	비고
1	공통	문화예술교육개론	2	2	2018년 1학기	None
2	공통	문화예술교육현장의 이해와 실습	2	3	2017년 2학기	None
3	연극	연극영화교과교육론	3	3	2018년 1학기	None
*/


CREATE TABLE departments (
	dept_id BIGINT, 
	dept_name TEXT, 
	org_id BIGINT, 
	dept_code TEXT, 
	phone TEXT, 
	fax TEXT, 
	floor_location TEXT, 
	description TEXT
)

/*
3 rows from departments table:
dept_id	dept_name	org_id	dept_code	phone	fax	floor_location	description
101	홍보담당관	2	D101	041-521-2080	041-521-2089	본관 8층	홍보업무 총괄 및 보도자료 관리
102	감사관	2	D102	041-521-2040	041-521-2049	본관 4층	감사업무 총괄 및 청렴윤리 관리
103	스마트도시추진과	2	D103	041-521-2205	041-521-2209	본관 7층	스마트도시 정책 및 AI산업 추진
*/


CREATE T

## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [14]:
# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = """
SELECT 분야
FROM art
WHERE 단과대학 = '디자인대학'
LIMIT 10;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT 분야
FROM art
WHERE 단과대학 = '디자인대학'
LIMIT 10;


결과:
[('디자인',), ('디자인',), ('디자인',), ('디자인',), ('디자인',), ('공예',), ('공예',)]


In [24]:
# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = """
SELECT art.분야, art."학과(전공)", art2."신청 교과목명"
FROM art
INNER JOIN art_2 art2 ON art.분야 = art2.분야
LIMIT 10;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT art.분야, art."학과(전공)", art2."신청 교과목명"
FROM art
INNER JOIN art_2 art2 ON art.분야 = art2.분야
LIMIT 10;


결과:
[('디자인', '커뮤니케이션디자인전공', '디자인 교육프로그램 개발'), ('디자인', '커뮤니케이션디자인전공', '디자인 교수학습방법(유아,초등,중등,일반)'), ('디자인', '커뮤니케이션디자인전공', '디자인 교육론'), ('디자인', '패션디자인전공', '디자인 교육프로그램 개발'), ('디자인', '패션디자인전공', '디자인 교수학습방법(유아,초등,중등,일반)'), ('디자인', '패션디자인전공', '디자인 교육론'), ('디자인', '텍스타일디자인전공', '디자인 교육프로그램 개발'), ('디자인', '텍스타일디자인전공', '디자인 교수학습방법(유아,초등,중등,일반)'), ('디자인', '텍스타일디자인전공', '디자인 교육론'), ('디자인', '스페이스디자인전공', '디자인 교육프로그램 개발')]


In [26]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT, SUM, AVG 등 사용

aggregation_query = """
SELECT 분야, COUNT(*) AS count
FROM art
GROUP BY 분야
ORDER BY count DESC;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT 분야, COUNT(*) AS count
FROM art
GROUP BY 분야
ORDER BY count DESC;


결과:
[('디자인', 5), ('공예', 2), ('영화', 1), ('연극', 1), ('사진', 1), ('만화·애니메이션', 1)]


## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [27]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    당신은 대학 학사 데이터(전공/학과, 교과목 인정) 전문 SQL 분석가입니다.
    사용자의 질문을 PostgreSQL SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [28]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "사진 분야가 신청 할 수 있는 교과목은 무엇이 있나요?"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: 사진 분야가 신청 할 수 있는 교과목은 무엇이 있나요?


생성된 SQL:
SELECT "신청 교과목명"
FROM art_2
WHERE "분야" = '사진'
ORDER BY "개설학년", "신청 교과목명";


실행 결과:
[('사진 교과교육론',), ('사진 교수학습방법(유아,초등,중등,일반)',), ('사진 교수학습프로그램 개발',)]


## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [29]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 art 데이터 분석 전문가입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문에 자연스럽게 답변하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [31]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "텍스타일디자인전공의 분야는 몇개인가요?"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: 텍스타일디자인전공의 분야는 몇개인가요?


[1] SQL 생성 중...
    SELECT COUNT(DISTINCT "분야") AS 분야_개수
FROM art
WHERE "학과(전공)" = '텍스타일디자인전공';

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...


답변:


텍스타일디자인전공의 분야는 **2개**입니다.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [32]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "디자인 분야의 학과는 무엇이 있나요?",
    "영화영상전공은 무슨 분야인가요?",
    "공통 분야의 인정 시기는 각각 언제인가요?",
    "공예 분야가 신청 가능한 교과목은 무엇이 있나요?",
    "3학점이 인정되는 교과목은 무엇이 있나요?"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: 디자인 분야의 학과는 무엇이 있나요?

[1] SQL 생성 중...
    SELECT DISTINCT "학과(전공)"
FROM art
WHERE "분야" = '디자인'
ORDER BY "학과(전공)";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


디자인 분야의 학과(전공)는 다음과 같습니다.

- 스페이스디자인전공
- 인더스트리얼디자인전공
- 커뮤니케이션디자인전공
- 텍스타일디자인전공
- 패션디자인전공

원하시면 각 전공이 어떤 공부를 하는지도 간단히 설명해드릴게요.


질문: 영화영상전공은 무슨 분야인가요?

[1] SQL 생성 중...
    SELECT "분야"
FROM art
WHERE "학과(전공)" = '영화영상전공'
LIMIT 1;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


영화영상전공은 **영화 분야**입니다.


질문: 공통 분야의 인정 시기는 각각 언제인가요?

[1] SQL 생성 중...
    SELECT "신청 교과목명", "인정시기"
FROM art_2
WHERE "분야" = '공통'
ORDER BY "신청 교과목명";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


공통 분야의 인정 시기는 다음과 같습니다.

- **문화예술교육개론**: **2018년 1학기**
- **문화예술교육현장의 이해와 실습**: **2017년 2학기**

원하시면 제가 이 내용을 표로도 정리해드릴게요.


질문: 공예 분야가 신청 가능한 교과목은 무엇이 있나요?

[1] SQL 생성 중...
    SELECT "신청 교과목명"
FROM art_2
WHERE "분야" = '공예'
ORDER BY "신청 교과목명";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


공예 분야에서 신청 가능한 교과목은 다음 3개입니다.

- 공예 교수학습방법
- 공예 교육론
- 공예 교육프로그램 개발

원하시면 제가 이 과목들을 표로도 정리해드릴게요.


질문: 3학점이 인정되는 교과목은 무엇이 있나요?

[1] SQL 생성 중...
    SELECT "신청 교과목명"
FROM art_2
WHERE "학점" = 3
ORDER BY "신청 교과목명";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


3학점이 인정되는 교과목은 다음과 같습니다.

- 연극영화교과교육론
- 연극영화교과교재및연구법

참고로 결과에 **연극영화교과교육론**이 두 번 나타났지만, 교과목명 기준으로 보면 위 두 과목입니다.

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용